# 分析数据,

In [ ]:
import padas as pd
import numpy as np
import os

def analyze_raw_data(input_file):
    if not os.path.exists(input_file):
        print(f"错误: 找不到文件 {input_file}")
        return

    print(f"正在读取 {input_file} 进行深度分析 ...")
    df = pd.read_csv(input_file)
    total_rows = len(df)
    print(f"总行数: {total_rows}")
    print("-" * 50)

    # 定义我们要检查的字段分组
    # z_cols = ['z', 'zErr']
    z_cols = ['z']
    mag_cols = ['psfMag_u', 'psfMag_g', 'psfMag_r', 'psfMag_i', 'psfMag_z']
    ext_cols = ['extinction_u', 'extinction_g', 'extinction_r', 'extinction_i', 'extinction_z']
    err_cols = ['psfMagErr_u', 'psfMagErr_g', 'psfMagErr_r', 'psfMagErr_i', 'psfMagErr_z']

    all_target_cols = z_cols + mag_cols + ext_cols + err_cols

    # ========================================================
    # 1. 基础缺失值与特殊值检查
    # ========================================================
    print("【1. 异常值与特殊占位符统计】")
    print(f"{'字段名':<15} | {'NaN数':<8} | {'-9999数':<8} | {'Neg/Zero数(仅Err)':<15} | {'Inf数':<8}")
    print("-" * 75)

    for col in all_target_cols:
        if col not in df.columns:
            print(f"警告: 列 {col} 不存在！")
            continue

        # 统计 NaN
        nan_count = df[col].isna().sum()

        # 统计 SDSS 常见错误码 -9999 (包括浮点数)
        # 注意：有时候也是 -1000 或 -999，这里主要抓 -9999
        placeholder_count = df[col].apply(lambda x: 1 if x == -9999 or x == -9999.0 else 0).sum()

        # 统计无穷大
        inf_count = np.isinf(df[col]).sum()

        # 针对 Error 和 zErr 统计非正数 (物理上 Error 必须 > 0)
        neg_zero_count = 0
        if 'Err' in col or col == 'zErr':
            neg_zero_count = (df[col] <= 0).sum()
        elif col == 'z':
            # 对于红移，只统计负数（虽然 z<0 物理上代表蓝移，但在类星体表中通常是错误）
            neg_zero_count = (df[col] < 0).sum()

        print(f"{col:<15} | {nan_count:<8} | {placeholder_count:<8} | {neg_zero_count:<15} | {inf_count:<8}")

    print("-" * 75)
    print("* 'Neg/Zero数': 对于 Err 字段统计 <=0，对于 z 字段统计 <0")
    print("\n")

    # ========================================================
    # 2. 详细统计分布 (Describe)
    # ========================================================
    print("【2. 字段统计分布概览 (过滤掉 -9999 后)】")
    # 为了让统计有意义，我们需要先把 -9999 和 inf 替换为 NaN，否则均值会被拉低
    df_stat = df.copy()
    df_stat.replace([-9999, -9999.0, np.inf, -np.inf], np.nan, inplace=True)

    # 转置输出，查看更方便
    desc = df_stat[all_target_cols].describe(percentiles=[0.01, 0.25, 0.5, 0.75, 0.95,0.99]).T
    # 设置显示格式
    pd.set_option('display.max_columns', None)
    pd.set_option('display.width', 1000)
    print(desc[['count', 'mean', 'min', '1%', '50%', '99%', 'max', 'std']])
    print("\n")

    # ========================================================
    # 3. 专家级深度诊断 (针对上一轮发现的 -6.0 问题)
    # ========================================================
    # print("【3. 深度诊断：zErr 分布特异性检查】")
    # # 检查 zErr 中小于 0 的具体数值有哪些
    # invalid_zerr = df['zErr'][df['zErr'] <= 0].unique()
    # if len(invalid_zerr) > 0:
    #     print(f"警告！发现 zErr 存在以下非正数值 (可能是错误代码): {np.sort(invalid_zerr)}")
    #     count_invalid = (df['zErr'] <= 0).sum()
    #     print(f"涉及行数: {count_invalid} (占比 {count_invalid/total_rows:.2%})")
    #     print("建议: 在排序去重前，必须删除这些行！")
    # else:
    #     print("通过: zErr 全部为正数。")

In [7]:
# 请确保路径正确
input_filename = '/mnt/d/SoftWare/PycharmProjects/AstroCLIP-main/dsm/script/sdss_quasar_dataset.csv'
analyze_raw_data(input_filename)

正在读取 /mnt/d/SoftWare/PycharmProjects/AstroCLIP-main/dsm/script/sdss_quasar_dataset.csv 进行深度分析 ...
总行数: 750414
--------------------------------------------------
【1. 异常值与特殊占位符统计】
字段名             | NaN数     | -9999数   | Neg/Zero数(仅Err) | Inf数    
---------------------------------------------------------------------------
z               | 0        | 0        | 10              | 0       
psfMag_u        | 0        | 1014     | 0               | 0       
psfMag_g        | 0        | 1015     | 0               | 0       
psfMag_r        | 0        | 1011     | 0               | 0       
psfMag_i        | 0        | 1011     | 0               | 0       
psfMag_z        | 0        | 1011     | 0               | 0       
extinction_u    | 0        | 999      | 0               | 0       
extinction_g    | 0        | 999      | 0               | 0       
extinction_r    | 0        | 999      | 0               | 0       
extinction_i    | 0        | 999      | 0               | 0       
extinctio

In [77]:
import pandas as pd

input_filename = './csv/dr18psf_ugriz_SNR_z_ext_clean.csv'
d=pd.read_csv(input_filename)
print(len(d))
d1=d[d['psfMagErr_u']<1e6]
d1_d=d1.describe(percentiles=[0.01, 0.25, 0.5, 0.95,0.99]).T
print(d1_d[['count', 'mean', 'min', '1%', '50%','99%', 'max', 'std']])

d2=d1[d1['psfMagErr_i']<1e6]
d2_d=d2.describe(percentiles=[0.01, 0.25, 0.5, 0.95,0.99]).T
print(d2_d[['count', 'mean', 'min', '1%', '50%','99%', 'max', 'std']])

d3=d2[d2['psfMagErr_z']<1e2]
d3_d=d3.describe(percentiles=[0.01, 0.25, 0.5, 0.95,0.99]).T
print(d3_d[['count', 'mean', 'min', '1%', '50%','99%', 'max', 'std']])

print(len(d1))
print(len(d2))
print(len(d3))

465313
                 count          mean           min            1%           50%           99%           max           std
mjd           465312.0  5.662643e+04  5.157800e+04  5.199300e+04  5.693300e+04  5.851500e+04  5.893200e+04  1.541536e+03
fiberID       465312.0  4.803449e+02  1.000000e+00  1.200000e+01  4.680000e+02  9.880000e+02  1.000000e+03  2.833280e+02
specObjID     465312.0  7.765049e+18  2.995037e+17  5.517837e+17  8.344124e+18  1.300878e+19  1.412694e+19  3.048378e+18
bestObjID     465312.0  1.237665e+18  1.237646e+18  1.237650e+18  1.237664e+18  1.237680e+18  1.237681e+18  8.665815e+12
z             465312.0  1.963350e+00  1.000002e+00  1.020012e+00  1.904082e+00  3.565272e+00  4.999749e+00  6.162732e-01
zErr          465312.0  4.852281e-04  8.100903e-09  1.169035e-04  4.549165e-04  9.729387e-04  9.999862e-04  2.121666e-04
run           465312.0  4.598142e+03  9.400000e+01  1.035000e+03  4.207000e+03  8.151000e+03  8.162000e+03  2.017671e+03
camcol        465312.0  3

In [78]:
output_filename="./csv/dr18psf_ugriz_SNR_z_ext_clean_err.csv"
d3.to_csv(output_filename, index=False)

In [66]:
import pandas as pd

input_filename = './csv/dr18psf_ugriz_SNR_z_ext_nnllkk_0_clean_err.csv'
d=pd.read_csv(input_filename)
print(len(d))
d1=d[d['psfMagErr_u']<1e6]
d1_d=d1.describe(percentiles=[0.01, 0.25, 0.5, 0.95,0.99]).T
print(d1_d[['count', 'mean', 'min', '1%', '50%','99%', 'max', 'std']])

d2=d1[d1['psfMagErr_i']<1e6]
d2_d=d2.describe(percentiles=[0.01, 0.25, 0.5, 0.95,0.99]).T
print(d2_d[['count', 'mean', 'min', '1%', '50%','99%', 'max', 'std']])

d3=d2[d2['psfMagErr_z']<1e2]
d3_d=d3.describe(percentiles=[0.01, 0.25, 0.5, 0.95,0.99]).T
print(d3_d[['count', 'mean', 'min', '1%', '50%','99%', 'max', 'std']])

print(len(d1))
print(len(d2))
print(len(d3))

465310
                 count          mean           min            1%           50%           99%           max           std
run           465310.0  4.598139e+03  9.400000e+01  1.035000e+03  4.207000e+03  8.151000e+03  8.162000e+03  2.017667e+03
camcol        465310.0  3.492794e+00  1.000000e+00  1.000000e+00  4.000000e+00  6.000000e+00  6.000000e+00  1.597176e+00
field         465310.0  1.789378e+02  1.100000e+01  1.600000e+01  1.450000e+02  6.110000e+02  9.840000e+02  1.321357e+02
specObjID     465310.0  7.765055e+18  2.995037e+17  5.517835e+17  8.344124e+18  1.300878e+19  1.412694e+19  3.048380e+18
bestObjID     465310.0  1.237665e+18  1.237646e+18  1.237650e+18  1.237664e+18  1.237680e+18  1.237681e+18  8.665798e+12
z             465310.0  1.963347e+00  1.000002e+00  1.020012e+00  1.904076e+00  3.565273e+00  4.999749e+00  6.162731e-01
zErr          465310.0  4.852276e-04  8.100903e-09  1.169034e-04  4.549158e-04  9.729391e-04  9.999862e-04  2.121669e-04
objID         465310.0  1

# 清洗数据,

In [75]:
import pandas as pd
import os
import numpy as np

def preprocess_catalog_advanced(input_file, output_file):
    if not os.path.exists(input_file):
        print(f"错误: 找不到文件 {input_file}")
        return

    print(f"正在读取 {input_file} ...")
    df = pd.read_csv(input_file)
    count_initial = len(df)

    # ==========================================
    # 1. 基础清洗：处理无效值标记 (inf, -9999)
    # ==========================================
    df.replace([np.inf, -np.inf], np.nan, inplace=True)
    df.replace(-9999, np.nan, inplace=True)
    df.replace(-9999.0, np.nan, inplace=True)

    # 这一步会删除包含 NaN 的行
    df_clean = df.dropna()

    count_after_dropna = len(df_clean)

    # ==========================================
    # 2. 物理清洗 (关键修复步骤！)
    # ==========================================
    # zErr 必须大于 0。任何小于等于 0 的 zErr 都是物理上无意义的错误标记。
    # 这会把 -6.0, -1.0, 0.0 等所有怪异数值全部剔除。
    df_clean = df_clean[df_clean['zErr'] > 0.0].copy()

    # # 顺便检查 z 是否在合理范围内 (防止 -9999 漏网之鱼或占位符)
    # # 对于类星体，z > 0 是必须的
    # df_clean = df_clean[df_clean['z'] > 0].copy()

    count_after_physics = len(df_clean)
    count_zErr = count_after_dropna-count_after_physics

    # ==========================================
    # 3. 排序与去重
    # ==========================================
    print("正在排序和去重...")
    # 现在剩下的 zErr 都是正数，最小的正数确实代表“精度最高”
    df_sorted = df_clean.sort_values(by='zErr', ascending=True)

    # 保留 objID 重复项中 zErr 最小的
    df_final = df_sorted.drop_duplicates(subset=['objID'], keep='first')

    # 获取最终数量
    count_final = len(df_final)

    # ==========================================
    # 4. 保存与报告
    # ==========================================
    df_final.to_csv(output_file, index=False)

    print("=" * 40)
    print("数据预处理报告 (修复版)")
    print("=" * 40)
    print(f"1. 原始数据总数        : {count_initial}")
    print(f"2. 去除空值后数量      : {count_after_dropna}")
    print(f"3. 去除物理无效值后数量 : {count_after_physics} (已剔除 zErr<=0，删除了 {count_zErr} 行)")
    print(f"4. 最终去重后数量      : {count_final}")
    print("-" * 40)
    print(f"共剔除无效/重复数据    : {count_initial - count_final}")
    print(f"结果已保存至          : {output_file}")

    # 额外验证：打印一下现在的 zErr 最小值，确保不再是负数
    min_zErr = df_final['zErr'].min()
    max_zErr = df_final['zErr'].max()
    print(f"验证: 当前最小 zErr 为 : {min_zErr}")
    print(f"验证: 当前最大 zErr 为 : {max_zErr}")

if __name__ == "__main__":
    input_filename = 'csv/dr18psf_ugriz_SNR_z_ext.csv'
    output_filename = 'csv/dr18psf_ugriz_SNR_z_ext_clean.csv'

    preprocess_catalog_advanced(input_filename, output_filename)

正在读取 csv/dr18psf_ugriz_SNR_z_ext.csv ...
正在排序和去重...
数据预处理报告 (修复版)
1. 原始数据总数        : 608451
2. 去除空值后数量      : 608451
3. 去除物理无效值后数量 : 604979 (已剔除 zErr<=0，删除了 3472 行)
4. 最终去重后数量      : 465313
----------------------------------------
共剔除无效/重复数据    : 143138
结果已保存至          : csv/dr18psf_ugriz_SNR_z_ext_clean.csv
验证: 当前最小 zErr 为 : 8.100903e-09
验证: 当前最大 zErr 为 : 0.0009999862


In [60]:
import pandas as pd

d=pd.read_csv('csv/data_1_clean.csv')
print(len(d))

671084


In [ ]:
if __name__ == "__main__":
    input_filename = 'csv/dr18psf_ugriz_SNR_z_ext_nnllkk.csv'
    output_filename = 'csv/dr18psf_ugriz_SNR_z_ext_clean.csv'

    preprocess_catalog_advanced(input_filename, output_filename)

# 合并两张表

In [17]:
import pandas as pd
import os
import numpy as np

def merge_and_finalize(file1, file2, output_file):
    # 1. 检查文件
    if not os.path.exists(file1):
        print(f"错误: 找不到文件 {file1}")
        return
    if not os.path.exists(file2):
        print(f"错误: 找不到文件 {file2}")
        return

    print("正在读取数据表...")
    # 读取两个 CSV
    df1 = pd.read_csv(file1)
    df2 = pd.read_csv(file2)

    print(f"表1行数: {len(df1)}")
    print(f"表2行数: {len(df2)}")

    # 2. 合并 (Concatenation)
    # 自动对齐列名，如果两表列名顺序不同也没关系
    df_merged = pd.concat([df1, df2], ignore_index=True)
    count_merged = len(df_merged)
    print(f"合并后初步总数: {count_merged}")

    # ==========================================
    # 3. 关键清洗步骤 (防止 -6.0 等异常值再次出现)
    # ==========================================
    # 处理缺失值标记
    df_merged.replace([np.inf, -np.inf], np.nan, inplace=True)
    df_merged.replace(-9999, np.nan, inplace=True)
    df_merged.replace(-9999.0, np.nan, inplace=True)

    # 剔除 NaN
    df_clean = df_merged.dropna()

    # 核心物理过滤：只保留 zErr > 0 的数据
    # 这步至关重要，防止排序时负值误差排在最前面
    df_clean = df_clean[df_clean['zErr'] > 0].copy()

    # 可选：如果你只想要特定红移范围 (例如 z > 0)
    df_clean = df_clean[df_clean['z'] > 0].copy()

    # ==========================================
    # 4. 排序与去重
    # ==========================================
    print("正在根据最佳红移误差进行去重...")

    # 按 zErr 升序排列 (最小的正误差排在第一位)
    df_sorted = df_clean.sort_values(by='zErr', ascending=True)

    # 对 objID 去重
    df_final = df_sorted.drop_duplicates(subset=['objID'], keep='first')

    count_final = len(df_final)

    # ==========================================
    # 5. 保存结果
    # ==========================================
    df_final.to_csv(output_file, index=False)

    print("=" * 40)
    print("合并与最终清洗完成")
    print("=" * 40)
    print(f"合并前总数据量 (Raw) : {len(df1) + len(df2)}")
    print(f"合并后总数据量 (Raw) : {count_merged}")
    print(f"清洗并去重后数据量   : {count_final}")
    print("-" * 40)
    print(f"共剔除数据 (含重复/坏值): {count_merged - count_final}")
    print(f"最终文件已保存为      : {output_file}")

    # 最终验证
    if count_final > 0:
        print(f"验证: 最终数据的 zErr 最小值: {df_final['zErr'].min()}")
        print(f"验证: 最终数据的 zErr 最大值: {df_final['zErr'].max()}")

if __name__ == "__main__":
    # 输入文件路径
    file_a = '../dbx/csv/hong_clean.csv'      # 刚才清洗过的表
    file_b = '../dbx/csv/dr18psf_ugriz_SNR_clean.csv'     # 需要合并的另一张表

    # 输出文件路径
    file_out = '../dbx/csv/merged_universe.csv'

    merge_and_finalize(file_a, file_b, file_out)

正在读取数据表...
表1行数: 413398
表2行数: 722458
合并后初步总数: 1135856
正在根据最佳红移误差进行去重...
合并与最终清洗完成
合并前总数据量 (Raw) : 1135856
合并后总数据量 (Raw) : 1135856
清洗并去重后数据量   : 747281
----------------------------------------
共剔除数据 (含重复/坏值): 388575
最终文件已保存为      : ../dbx/csv/merged_universe.csv
验证: 最终数据的 zErr 最小值: 8.100903e-09
验证: 最终数据的 zErr 最大值: 2299.29
